<a id='9c-14'></a>

## 9c-14. 🧩 Pattern 14: itertools essentials — LC 46, 39, 78

---

```
PROBLEM:  Building permutations, combinations, products, or grouped sequences
          by hand produces nested loops and off-by-one bugs.

APPROACH: itertools is a lazy iterator factory — it generates items on demand,
          uses O(1) memory regardless of output size. Wrap in list() only when
          you need all results at once.

SLOW MOTION TRACE:
  # permutations — all orderings
  permutations([1,2,3], 2)  ->  (1,2),(1,3),(2,1),(2,3),(3,1),(3,2)

  # combinations — subsets, no repeats, order doesn't matter
  combinations([1,2,3], 2)  ->  (1,2),(1,3),(2,3)

  # combinations_with_replacement — repeats allowed
  combinations_with_replacement([1,2], 2)  ->  (1,1),(1,2),(2,2)

  # product — cartesian product, like nested for-loops
  product([0,1], repeat=2)  ->  (0,0),(0,1),(1,0),(1,1)

  # chain — flatten iterables without creating intermediate lists
  chain([1,2], [3,4], [5])  ->  1,2,3,4,5

  # groupby — group consecutive elements sharing a key (MUST be pre-sorted)
  groupby([1,1,2,2,3], key=lambda x:x)
    -> (1,[1,1]), (2,[2,2]), (3,[3])

  # accumulate — running totals (prefix sums without a manual loop)
  accumulate([1,2,3,4])  ->  1, 3, 6, 10

KEY INSIGHT: accumulate replaces the manual prefix-sum loop; chain replaces
             [item for sublist in nested for item in sublist] flatten patterns.

TIME / SPACE: All itertools are lazy — O(1) memory per next() call.
              Total time is O(output_size) when you consume all items.
```


In [ ]:
from itertools import (
    permutations, combinations, combinations_with_replacement,
    product, chain, groupby, accumulate
)
from typing import List


# ── permutations ──────────────────────────────────────────────────────────────
# Think: all ways to arrange items — like seating arrangements
perms = list(permutations([1, 2, 3]))
print(f"permutations([1,2,3]): {perms}")     # 6 = 3! arrangements

perms_2 = list(permutations([1, 2, 3], 2))
print(f"permutations(r=2): {perms_2}")        # 6 = 3P2


# ── combinations ─────────────────────────────────────────────────────────────
# Think: picking teams — order doesn't matter, no repeats
combos = list(combinations([1, 2, 3, 4], 2))
print(f"combinations(r=2): {combos}")         # 6 = 4C2

combos_rep = list(combinations_with_replacement([1, 2], 2))
print(f"with_replacement: {combos_rep}")      # (1,1),(1,2),(2,2)


# ── product — cartesian product ───────────────────────────────────────────────
# Think: all combinations of two independent choices
binary_pairs = list(product([0, 1], repeat=2))
print(f"product([0,1], repeat=2): {binary_pairs}")   # truth table

suits  = ['♠', '♥']
values = ['A', 'K']
deck   = list(product(suits, values))
print(f"mini deck: {deck}")                    # [('♠','A'),('♠','K'),('♥','A'),('♥','K')]


# ── chain — lazy flatten ──────────────────────────────────────────────────────
# Think: connecting garden hoses — one stream from many sources
flat = list(chain([1, 2], [3, 4], [5]))
print(f"chain: {flat}")                        # [1,2,3,4,5]

# chain.from_iterable — flatten a list-of-lists
nested = [[1, 2], [3, 4], [5]]
print(list(chain.from_iterable(nested)))       # [1,2,3,4,5]


# ── accumulate — prefix sums ──────────────────────────────────────────────────
# Think: running bank balance — each step adds to the total so far
nums = [1, 2, 3, 4, 5]
prefix = list(accumulate(nums))
print(f"prefix sums: {prefix}")                # [1,3,6,10,15]

# prefix[j] - prefix[i] = sum of nums[i+1..j]  — O(1) range sum query
print(f"sum [1..3] = {prefix[3] - prefix[0]}")  # sum of [2,3,4] = 9


# ── groupby — group consecutive equal elements ────────────────────────────────
# CRITICAL: input MUST be sorted by the key first — groupby only sees neighbors
data = sorted([3, 1, 1, 2, 2, 3], key=lambda x: x)
for key, group in groupby(data, key=lambda x: x):
    print(f"  key={key}  group={list(group)}")
# key=1 group=[1,1]
# key=2 group=[2,2]
# key=3 group=[3,3]


# ── LC 78 — Subsets using combinations ───────────────────────────────────────
def subsets(nums: List[int]) -> List[List[int]]:
    """
    LC 78 — Subsets.
    Approach: itertools.combinations for every size 0..n.
    Args:
        nums (List[int]): distinct integers.
    Returns:
        List[List[int]]: all possible subsets (power set).
    Time:  O(2^n) — 2^n subsets, each up to size n
    Space: O(2^n) — storing all subsets
    """
    result = []
    for size in range(len(nums) + 1):        # size 0 (empty set) through n (full set)
        for combo in combinations(nums, size):
            result.append(list(combo))
    return result

def test_harness_subsets(fn):
    tests = [
        ([1, 2, 3], [[], [1], [2], [3], [1,2], [1,3], [2,3], [1,2,3]]),
        ([0], [[], [0]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        got_sorted = sorted(sorted(g) for g in got)
        exp_sorted = sorted(sorted(g) for g in expected)
        status = "PASSED" if got_sorted == exp_sorted else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got_sorted == exp_sorted)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_subsets(subsets)
print("subsets defined.")


# ── LC 46 — Permutations (pure itertools version) ────────────────────────────
def permute(nums: List[int]) -> List[List[int]]:
    """
    LC 46 — Permutations.
    Approach: itertools.permutations — all n! orderings.
    Args:
        nums (List[int]): distinct integers.
    Returns:
        List[List[int]]: all permutations.
    Time:  O(n! * n) — n! permutations, each of length n
    Space: O(n! * n) — storing all permutations
    """
    return [list(p) for p in permutations(nums)]

def test_harness_permute(fn):
    tests = [
        ([1, 2, 3], [[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]),
        ([0, 1], [[0,1],[1,0]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        got_sorted = sorted(got)
        exp_sorted = sorted(expected)
        status = "PASSED" if got_sorted == exp_sorted else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got_sorted == exp_sorted)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_permute(permute)
print("permute defined.")

# Simplicity and clarity is Gold
